# PRISM Rebuttal round 2 - Reviewer tp5b follow-up

Four analyses, all CPU, all from data already in `results_v2/`.

| # | what the reviewer said | what this notebook produces |
|---|---|---|
| 1 | "in the supplementary I find only the original analysis at a fixed threshold of 0.5 and no corrected breaking-point table" | breaking points at F1 thresholds 0.4 to 0.8, the table our reply referred to |
| 2 | "sweeping the regularization parameter tests hyperparameter sensitivity, not variability caused by selecting very few training samples" | the measurement they actually asked for: between-seed standard deviation as a function of label fraction |
| 3 | "the reported bootstrap analysis is performed after averaging over seeds" | rank correlations recomputed by resampling seeds rather than averaging them away |
| 4 | "I would prefer the individual in-domain AUROC, OOD AUROC, and performance drop to be reported alongside, or instead of, the composite CRI" | the three components tabulated per model |

The reviewer is right on points 2 and 3 as stated: our sweep varied a
hyperparameter while holding the subset fixed, which is a different question
from subset variability, and our bootstrap resampled datasets after collapsing
seeds. Both are measured properly here, and we report whatever the numbers
show.

Point 1 is a claim our reply made that the repository did not support. This
notebook produces the table so it does.

Writes to `results_v2/`. Runtime a few minutes.

In [1]:
import os, warnings
import numpy as np, pandas as pd
from scipy.stats import spearmanr
from google.colab import drive

warnings.filterwarnings('ignore')
drive.mount('/content/drive')

BASE = '/content/drive/MyDrive/PRISM'
RES  = f'{BASE}/results_v2'

ind = pd.read_csv(f'{RES}/indomain_all_v2.csv')
ood = pd.read_csv(f'{RES}/ood_all_v2.csv')

MODELS = ['CLIP','PLIP','CONCH','VIRCHOW2','UNI','GigaPath','H-Optimus-0','MIDNIGHT']
DATASETS = ['PCam','BRACS','CRC','MHIST','LungHist700','SPIDER-Breast']
FRACTIONS = [0.01, 0.05, 0.10, 0.25, 0.50, 1.00]
OOD_PAIRS = [('PCam','MHIST'), ('MHIST','PCam'), ('CRC','BRACS'), ('BRACS','CRC')]

print(f'in-domain {len(ind)} rows, OOD {len(ood)} rows')
print('seeds per cell:', ind.groupby(['model','dataset','fraction']).size().unique())

Mounted at /content/drive
in-domain 864 rows, OOD 576 rows
seeds per cell: [3]


## 1. Breaking points across F1 thresholds

The reviewer could not verify a table our reply referred to. Here it is.

A breaking point is the smallest label fraction at which mean macro-F1 reaches
the threshold. `>1.00` means the threshold is never reached, even at full
supervision.

In [2]:
THRESHOLDS = [0.4, 0.5, 0.6, 0.7, 0.8]

def breaking_point(model, dataset, thr, col='f1_macro'):
    s = (ind[(ind.model == model) & (ind.dataset == dataset)]
         .groupby('fraction')[col].mean().sort_index())
    hit = s[s >= thr]
    return float(hit.index[0]) if len(hit) else np.nan

rows = []
for ds in DATASETS:
    for m in MODELS:
        r = dict(dataset=ds, model=m)
        for thr in THRESHOLDS:
            r[f'thr_{thr}'] = breaking_point(m, ds, thr)
        rows.append(r)

bp = pd.DataFrame(rows)
bp.to_csv(f'{RES}/breaking_points_by_threshold.csv', index=False)

for ds in DATASETS:
    sub = bp[bp.dataset == ds].set_index('model')[
        [f'thr_{t}' for t in THRESHOLDS]]
    sub.columns = [f'F1>={t}' for t in THRESHOLDS]
    print(f'\n=== {ds} : smallest label fraction reaching each threshold ===')
    print(sub.fillna('>1.00').to_string())

print('\n\n=== How much does the threshold choice move the breaking point? ===')
sens = []
for ds in DATASETS:
    sub = bp[bp.dataset == ds]
    for m in MODELS:
        r = sub[sub.model == m].iloc[0]
        vals = [r[f'thr_{t}'] for t in THRESHOLDS if not np.isnan(r[f'thr_{t}'])]
        if len(vals) >= 2:
            sens.append(dict(dataset=ds, model=m,
                             lo=min(vals), hi=max(vals), span=max(vals)-min(vals)))
sens = pd.DataFrame(sens)
print(sens.groupby('dataset')[['span']].agg(['mean','max']).round(3).to_string())
print('\nA span of 0 means the breaking point is the same at every threshold '
      'tested;\nlarger spans mean the reported breaking point depends on where '
      'the line is drawn.')
print('\nSaved -> breaking_points_by_threshold.csv')


=== PCam : smallest label fraction reaching each threshold ===
             F1>=0.4  F1>=0.5  F1>=0.6  F1>=0.7  F1>=0.8
model                                                   
CLIP            0.01     0.01     0.01     0.01     0.01
PLIP            0.01     0.01     0.01     0.01     0.01
CONCH           0.01     0.01     0.01     0.01     0.01
VIRCHOW2        0.01     0.01     0.01     0.01     0.01
UNI             0.01     0.01     0.01     0.01     0.01
GigaPath        0.01     0.01     0.01     0.01     0.01
H-Optimus-0     0.01     0.01     0.01     0.01     0.01
MIDNIGHT        0.01     0.01     0.01     0.01     0.01

=== BRACS : smallest label fraction reaching each threshold ===
             F1>=0.4 F1>=0.5 F1>=0.6 F1>=0.7 F1>=0.8
model                                               
CLIP            1.00   >1.00   >1.00   >1.00   >1.00
PLIP            0.25   >1.00   >1.00   >1.00   >1.00
CONCH           0.05     0.1   >1.00   >1.00   >1.00
VIRCHOW2        0.05     0.1   >1.00

## 2. Between-seed variability, the measurement actually requested

Our regularisation sweep varied C while holding the training subset fixed. That
measures hyperparameter sensitivity. The reviewer's hypothesis is about
variability introduced by *which* few samples are drawn, which is measured by
the spread across seeds at fixed C.

We report it directly and without selection: standard deviation across the
three seeds, per (model, dataset), as a function of label fraction.

In [3]:
def seed_spread(df, metric, keys=('model','dataset')):
    g = df.groupby(list(keys) + ['fraction'])[metric]
    return g.agg(['mean','std','min','max']).reset_index()

print('=== Between-seed standard deviation by label fraction ===')
print('(mean over all model-dataset cells)\n')
for metric in ['auroc', 'f1_macro', 'ece_fixed', 'ece_scaled_fixed']:
    s = seed_spread(ind, metric)
    t = s.groupby('fraction')['std'].agg(['mean','max']).round(4)
    print(f'--- {metric} ---')
    print(t.to_string())
    print()

print('=== AUROC seed-std per dataset ===')
s = seed_spread(ind, 'auroc')
print(s.pivot_table(index='dataset', columns='fraction', values='std')
       .round(4).to_string())

print('\n=== Side by side: seed spread against C spread, AUROC ===')
try:
    c_ab = pd.read_csv(f'{RES}/c_ablation.csv')
    c_sp = (c_ab.groupby(['dataset','fraction','model'])['auroc']
                .agg(lambda v: v.max()-v.min())
                .groupby(level=[0,1]).mean().rename('C_spread'))
    seed_sd = (seed_spread(ind, 'auroc')
               .groupby(['dataset','fraction'])['std'].mean().rename('seed_std'))
    cmp = pd.concat([seed_sd, c_sp], axis=1).dropna()
    cmp = cmp[cmp.index.get_level_values('dataset')
              .isin(c_ab['dataset'].unique())]
    cmp['ratio'] = cmp['seed_std'] / cmp['C_spread']
    print(cmp.round(4).to_string())
    print('\nThese answer different questions and we now say so: the C spread is '
          'hyperparameter\nsensitivity, the seed std is subset-selection '
          'variability. Report both.')
except FileNotFoundError:
    print('c_ablation.csv not found')

seed_spread(ind, 'auroc').to_csv(f'{RES}/seed_variability_auroc.csv', index=False)
seed_spread(ind, 'ece_scaled_fixed').to_csv(f'{RES}/seed_variability_ece.csv',
                                            index=False)
print('\nSaved -> seed_variability_auroc.csv, seed_variability_ece.csv')

=== Between-seed standard deviation by label fraction ===
(mean over all model-dataset cells)

--- auroc ---
            mean     max
fraction                
0.01      0.0197  0.0845
0.05      0.0153  0.0669
0.10      0.0062  0.0228
0.25      0.0051  0.0236
0.50      0.0027  0.0138
1.00      0.0000  0.0000

--- f1_macro ---
            mean     max
fraction                
0.01      0.0175  0.1215
0.05      0.0106  0.0931
0.10      0.0119  0.0686
0.25      0.0130  0.0737
0.50      0.0075  0.0592
1.00      0.0000  0.0000

--- ece_fixed ---
            mean     max
fraction                
0.01      0.0203  0.1126
0.05      0.0130  0.0550
0.10      0.0103  0.0484
0.25      0.0091  0.0550
0.50      0.0058  0.0502
1.00      0.0000  0.0000

--- ece_scaled_fixed ---
            mean     max
fraction                
0.01      0.0137  0.0630
0.05      0.0125  0.0651
0.10      0.0093  0.0395
0.25      0.0070  0.0333
0.50      0.0058  0.0301
1.00      0.0000  0.0000

=== AUROC seed-std per data

### 2b. Does subset variability explain the decoupling?

Seed spread being largest at 1% labels is expected and does not by itself
decide the question. The question is whether the *rank correlation* between
discrimination and calibration is explained by that noise. We test it directly:
recompute the correlation within each individual seed, and see whether the
low-label pattern is present seed by seed or only after averaging.

In [4]:
def rho_at(fraction, seed=None, ece_col='ece_scaled_fixed'):
    d = ind[ind.fraction == fraction]
    if seed is not None:
        d = d[d.seed == seed]
    out = []
    for ds in DATASETS:
        s = d[d.dataset == ds].groupby('model')[['auroc', ece_col]].mean().dropna()
        if len(s) < 3:
            continue
        ra = s['auroc'].rank(ascending=False)
        rc = s[ece_col].rank(ascending=True)
        out += list(zip(ra.values, rc.values))
    if len(out) < 4:
        return np.nan
    a, c = zip(*out)
    return spearmanr(a, c).correlation

SEEDS = sorted(ind['seed'].unique())
print('Pooled rank correlation, computed within each seed separately\n')
print(f"{'frac':>6} " + ''.join(f'{f"seed {s}":>10}' for s in SEEDS)
      + f"{'mean':>10}{'averaged':>10}")
print('-' * 60)
for f in FRACTIONS:
    per = [rho_at(f, s) for s in SEEDS]
    print(f'{f:>6.2f} ' + ''.join(f'{v:>10.3f}' for v in per)
          + f'{np.nanmean(per):>10.3f}{rho_at(f):>10.3f}')

print('\nIf the low-label pattern appears in every individual seed, it is not '
      'an artefact\nof averaging. If it appears only in the averaged column, '
      'the reviewer is right.')

Pooled rank correlation, computed within each seed separately

  frac    seed 42  seed 123  seed 456      mean  averaged
------------------------------------------------------------
  0.01     -0.167    -0.254    -0.099    -0.173    -0.238
  0.05      0.258     0.333     0.155     0.249     0.389
  0.10      0.500     0.456     0.163     0.373     0.409
  0.25      0.183     0.421     0.345     0.316     0.444
  0.50      0.456     0.540     0.492     0.496     0.476
  1.00      0.516     0.516     0.516     0.516     0.516

If the low-label pattern appears in every individual seed, it is not an artefact
of averaging. If it appears only in the averaged column, the reviewer is right.


## 3. Bootstrap that resamples seeds rather than averaging them

Our earlier interval resampled datasets after collapsing the three seeds to a
mean. This version resamples at the level the data was generated: a bootstrap
draw picks datasets with replacement *and* picks one seed per cell, so
seed-level noise propagates into the interval instead of being averaged out
first.

In [5]:
def rho_from_draw(fraction, dsets, seed_choice, rng, ece_col='ece_scaled_fixed'):
    out = []
    for ds in dsets:
        s = ind[(ind.dataset == ds) & (ind.fraction == fraction)]
        if s.empty:
            continue
        picked = []
        for m in s['model'].unique():
            cell = s[s.model == m]
            row = cell[cell.seed == seed_choice[(ds, m)]]
            if len(row):
                picked.append((m, row['auroc'].iloc[0], row[ece_col].iloc[0]))
        if len(picked) < 3:
            continue
        df = pd.DataFrame(picked, columns=['model','auroc','ece']).set_index('model')
        ra = df['auroc'].rank(ascending=False)
        rc = df['ece'].rank(ascending=True)
        out += list(zip(ra.values, rc.values))
    if len(out) < 4:
        return np.nan
    a, c = zip(*out)
    return spearmanr(a, c).correlation


def bootstrap_seed_level(fraction, n_boot=2000, seed=0):
    rng = np.random.default_rng(seed)
    dsets = np.array(DATASETS)
    models = ind['model'].unique()
    boots = []
    for _ in range(n_boot):
        pick = rng.choice(dsets, size=len(dsets), replace=True)
        choice = {(ds, m): rng.choice(SEEDS) for ds in dsets for m in models}
        r = rho_from_draw(fraction, pick, choice, rng)
        if not np.isnan(r):
            boots.append(r)
    lo, hi = np.percentile(boots, [2.5, 97.5])
    return float(np.mean(boots)), float(lo), float(hi)


print('Pooled Spearman rho, bootstrap resampling datasets AND seeds\n')
print(f"{'frac':>6} {'rho':>8} {'CI low':>9} {'CI high':>9}   excludes 0")
print('-' * 48)
res = {}
for f in FRACTIONS:
    r, lo, hi = bootstrap_seed_level(f)
    res[f] = (r, lo, hi)
    star = 'yes' if (lo > 0 or hi < 0) else 'no'
    print(f'{f:>6.2f} {r:>8.3f} {lo:>9.3f} {hi:>9.3f}   {star}')

# contrast between the extremes, same resampling scheme
rng = np.random.default_rng(1)
dsets = np.array(DATASETS)
models = ind['model'].unique()
diffs = []
for _ in range(2000):
    pick = rng.choice(dsets, size=len(dsets), replace=True)
    choice = {(ds, m): rng.choice(SEEDS) for ds in dsets for m in models}
    a = rho_from_draw(0.01, pick, choice, rng)
    b = rho_from_draw(1.00, pick, choice, rng)
    if not (np.isnan(a) or np.isnan(b)):
        diffs.append(b - a)
lo, hi = np.percentile(diffs, [2.5, 97.5])
print(f'\nrho(100%) - rho(1%) = {np.mean(diffs):.3f}   95% CI [{lo:.3f}, {hi:.3f}]'
      + ('   SIGNIFICANT' if lo > 0 else '   not significant'))
print('\nCompare with the seed-averaged version reported earlier: '
      'Δρ = 0.75, CI [0.19, 1.27].')

pd.DataFrame([dict(fraction=f, rho=v[0], ci_lo=v[1], ci_hi=v[2])
              for f, v in res.items()]).to_csv(
    f'{RES}/rank_correlations_seedlevel.csv', index=False)
print('Saved -> rank_correlations_seedlevel.csv')

Pooled Spearman rho, bootstrap resampling datasets AND seeds

  frac      rho    CI low   CI high   excludes 0
------------------------------------------------
  0.01   -0.105    -0.540     0.357   no
  0.05    0.241    -0.218     0.651   no
  0.10    0.338    -0.071     0.698   no
  0.25    0.362     0.000     0.659   no
  0.50    0.492     0.048     0.790   yes
  1.00    0.518     0.266     0.722   yes

rho(100%) - rho(1%) = 0.616   95% CI [0.075, 1.127]   SIGNIFICANT

Compare with the seed-averaged version reported earlier: Δρ = 0.75, CI [0.19, 1.27].
Saved -> rank_correlations_seedlevel.csv


## 4. CRI components reported separately

The reviewer asked for in-domain AUROC, OOD AUROC and the performance drop
alongside or instead of the composite. Here they are per model, with the
composite in the last column so the two can be compared.

In [6]:
id100 = ind[ind.fraction == 1.0].groupby(['model','dataset'])['auroc'].mean()

rows = []
for m in MODELS:
    id_mean = id100.loc[m].mean() if m in id100.index.get_level_values(0) else np.nan
    pair_rows = []
    for src, tgt in OOD_PAIRS:
        o = ood[(ood.model == m) & (ood.pair == f'{src}->{tgt}') &
                (ood.fraction == 1.0)]['auroc'].mean()
        try:
            i = id100.loc[(m, src)]
        except KeyError:
            continue
        if np.isfinite(o) and i:
            pair_rows.append(dict(pair=f'{src}->{tgt}', id_auroc=i, ood_auroc=o,
                                  drop=i - o, ratio=min(o / i, 1.0)))
    if not pair_rows:
        continue
    pr = pd.DataFrame(pair_rows)
    rows.append(dict(
        model=m,
        id_auroc_mean=id_mean,
        ood_auroc_mean=pr['ood_auroc'].mean(),
        drop_mean=pr['drop'].mean(),
        drop_worst=pr['drop'].max(),
        ood_stability=pr['ratio'].mean()))

comp = pd.DataFrame(rows).set_index('model')

# composite alongside
cri = {}
for m in MODELS:
    s = ind[(ind.model == m) & (ind.fraction == 1.0)]
    if s.empty or m not in comp.index:
        continue
    a = s.groupby('dataset')['auroc'].mean().mean()
    e = float(np.clip(s.groupby('dataset')['ece_scaled_fixed'].mean().mean(), 0, 1))
    cri[m] = a * (1 - e) * comp.loc[m, 'ood_stability']
comp['cri'] = pd.Series(cri)

print('=== CRI components, at full supervision ===\n')
print(comp.round(4).sort_values('ood_auroc_mean', ascending=False).to_string())

print('\n=== Rankings by each component ===')
rk = pd.DataFrame({
    'by_id_auroc':  comp['id_auroc_mean'].rank(ascending=False),
    'by_ood_auroc': comp['ood_auroc_mean'].rank(ascending=False),
    'by_drop':      comp['drop_mean'].rank(ascending=True),
    'by_stability': comp['ood_stability'].rank(ascending=False),
    'by_cri':       comp['cri'].rank(ascending=False),
}).astype(int)
print(rk.to_string())

from scipy.stats import kendalltau
print('\nKendall tau between component rankings and the composite:')
for c in ['by_id_auroc','by_ood_auroc','by_drop','by_stability']:
    print(f"  {c:<14} vs by_cri: "
          f"{kendalltau(rk[c], rk['by_cri']).correlation:+.3f}")

print('\n=== Per-pair detail ===')
detail = []
for m in MODELS:
    for src, tgt in OOD_PAIRS:
        o = ood[(ood.model == m) & (ood.pair == f'{src}->{tgt}') &
                (ood.fraction == 1.0)]['auroc'].mean()
        try:
            i = id100.loc[(m, src)]
        except KeyError:
            continue
        detail.append(dict(model=m, pair=f'{src}->{tgt}',
                           id_auroc=round(i,4), ood_auroc=round(o,4),
                           drop=round(i-o,4)))
det = pd.DataFrame(detail)
print(det.pivot_table(index='model', columns='pair',
                      values='drop').round(3).to_string())

comp.to_csv(f'{RES}/cri_components.csv')
det.to_csv(f'{RES}/cri_components_per_pair.csv', index=False)
print('\nSaved -> cri_components.csv, cri_components_per_pair.csv')

=== CRI components, at full supervision ===

             id_auroc_mean  ood_auroc_mean  drop_mean  drop_worst  ood_stability     cri
model                                                                                   
VIRCHOW2            0.9622          0.6624     0.2889      0.4012         0.7001  0.6425
CLIP                0.8968          0.5748     0.3205      0.4326         0.6433  0.5410
H-Optimus-0         0.9537          0.5703     0.3704      0.4257         0.6061  0.5535
GigaPath            0.9540          0.5699     0.3714      0.4003         0.6055  0.5516
PLIP                0.9235          0.5426     0.3667      0.4223         0.5974  0.5202
CONCH               0.9468          0.4993     0.4309      0.6647         0.5418  0.4891
UNI                 0.9540          0.4890     0.4524      0.5367         0.5214  0.4694
MIDNIGHT            0.9198          0.4877     0.4082      0.5461         0.5501  0.4755

=== Rankings by each component ===
             by_id_auroc  by_

## 5. What to do with these

**1.** The threshold table now exists and the reply can point at it. If the
breaking points move substantially with the threshold, say so; that is the
honest reading and it supports the reviewer's caution rather than ours.

**2 and 2b.** Report the seed spread whatever it shows. If the within-seed
correlations reproduce the low-label pattern individually, the decoupling is
not an averaging artefact and we can say so with this table. If they do not,
the claim needs bounding.

**3.** Report the seed-level interval next to the earlier one. If it is wider
and no longer excludes zero, that is the number to quote from now on.

**4.** Add the component table to the manuscript next to the composite. The
reviewer's preference is reasonable and costs nothing: a reader who distrusts
the composite can read the three axes directly.